In [1]:
from rich.traceback import install; install()
from pathlib import Path
import pandas as pd 
from id_mapping import admission_source_mapping, discharge_disposition_mapping, admission_type_mapping, payer_mapping
from helper import *

In [2]:
# load data
data_dir = Path('../data')
df = pd.read_csv(data_dir / "diabetic_data.csv")
admission_type_code = pd.read_csv(data_dir / "IDS_mapping.csv", nrows=8)
discharge_disposition_code = pd.read_csv(data_dir / "IDS_mapping.csv", skiprows=10, nrows=30)
admission_source_code = pd.read_csv(data_dir / "IDS_mapping.csv", skiprows=42)

In [3]:
# Apply mappings
df['admission_type_grouped'] = df['admission_type_id'].map(admission_source_mapping)
df['discharge_disposition_grouped'] = df['discharge_disposition_id'].map(discharge_disposition_mapping)
df['admission_source_grouped'] = df['admission_source_id'].map(admission_source_mapping)

In [4]:
df.discharge_disposition_grouped.value_counts()

discharge_disposition_grouped
HOME                      73244
TRANSFERRED_POST_ACUTE    17222
UNKNOWN                    4680
TRANSFERRED_ACUTE_CARE     3317
DECEASED                   2423
LEFT_AMA                    623
PSYCHIATRIC                 139
STILL_IN_SYSTEM             112
NEWBORN_SPECIAL               6
Name: count, dtype: int64

In [5]:
df.admission_source_grouped.value_counts()

admission_source_grouped
EMERGENCY    57510
REFERRAL     30856
UNKNOWN       7067
TRANSFER      6328
NEWBORN          5
Name: count, dtype: int64

In [6]:
df.admission_type_grouped.value_counts()

admission_type_grouped
REFERRAL     91339
TRANSFER     10086
EMERGENCY      341
Name: count, dtype: int64

In [7]:
df_clean = df[
    ~df.discharge_disposition_grouped.isin(
        ['DECREASED', 'NEWBORN_SPECIAL', 'STILL_IN_SYSTEM']
    ) &
    ~df.admission_source_grouped.isin(['NEWBORN'])
]

df_clean = df_clean.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'race', 'encounter_id'])

In [8]:
df_clean

,patient_nbr,gender,age,weight,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped
0,8222157,Female,[0-10),?,1,?,Pediatrics-Endocrinology,41,0,1,...,No,No,No,No,No,No,NO,TRANSFER,UNKNOWN,REFERRAL
1,55629189,Female,[10-20),?,3,?,?,59,0,18,...,No,No,No,No,Ch,Yes,>30,REFERRAL,HOME,EMERGENCY
2,86047875,Female,[20-30),?,2,?,?,11,5,13,...,No,No,No,No,No,Yes,NO,REFERRAL,HOME,EMERGENCY
3,82442376,Male,[30-40),?,2,?,?,44,1,16,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,EMERGENCY
4,42519267,Male,[40-50),?,1,?,?,51,0,8,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,EMERGENCY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,100162476,Male,[70-80),?,3,MC,?,51,0,16,...,No,No,No,No,Ch,Yes,>30,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY
101762,74694222,Female,[80-90),?,5,MC,?,33,3,18,...,No,No,No,No,No,Yes,NO,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER
101763,41088789,Male,[70-80),?,1,MC,?,53,0,9,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,EMERGENCY
101764,31693671,Female,[80-90),?,10,MC,Surgery-General,45,2,21,...,No,No,No,No,Ch,Yes,NO,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY


In [9]:
df_clean.weight.value_counts() # I am going to drop weight column

weight
?            98446
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

In [10]:
(df_clean.weight == "?").mean() # 97% missing value

0.9685467764627176

In [11]:
df_clean = df_clean.drop(columns='weight')

In [12]:
df_clean.age.value_counts()

age
[70-80)     26034
[60-70)     22458
[50-60)     17245
[80-90)     17170
[40-50)      9671
[30-40)      3772
[90-100)     2788
[20-30)      1654
[10-20)       690
[0-10)        161
Name: count, dtype: int64

In [13]:
(df_clean.age.isin(['[0-10)', '[10-20)'])).mean() # drop the children and teen population 

0.00837244079769389

In [14]:
df_clean = df_clean[~df_clean.age.isin(['[0-10)', '[10-20)'])]

In [15]:
df_clean.payer_code.value_counts()

payer_code
?     39569
MC    32418
HM     6238
SP     4970
BC     4611
MD     3456
CP     2509
UN     2432
CM     1932
OG     1028
PO      588
DM      532
CH      144
WC      135
OT       95
MP       79
SI       55
FR        1
Name: count, dtype: int64

In [16]:
# reduce the number of categories in payer code, don't drop yet
df_clean.payer_code = df_clean.payer_code.map(payer_mapping)

In [17]:
df_clean.medical_specialty.value_counts() # also drop this because 50% empty and I don't think how the medical speciality of the admitting doctor matters here when we have the diagnosis to cover that point. Also there are too many categories

medical_specialty
?                                   49625
InternalMedicine                    14584
Emergency/Trauma                     7522
Family/GeneralPractice               7403
Cardiology                           5351
                                    ...  
Perinatology                            1
Neurophysiology                         1
Pediatrics-Pulmonology                  1
Psychiatry-Addictive                    1
Surgery-PlasticwithinHeadandNeck        1
Name: count, Length: 64, dtype: int64

In [18]:
df_clean = df_clean.drop(columns='medical_specialty')

In [19]:
df_clean

,patient_nbr,gender,age,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped
2,86047875,Female,[20-30),2,Unknown,11,5,13,2,0,...,No,No,No,No,No,Yes,NO,REFERRAL,HOME,EMERGENCY
3,82442376,Male,[30-40),2,Unknown,44,1,16,0,0,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,EMERGENCY
4,42519267,Male,[40-50),1,Unknown,51,0,8,0,0,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,EMERGENCY
5,82637451,Male,[50-60),3,Unknown,31,6,16,0,0,...,No,No,No,No,No,Yes,>30,REFERRAL,HOME,REFERRAL
6,84259809,Male,[60-70),4,Unknown,70,1,21,0,0,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,REFERRAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,100162476,Male,[70-80),3,Government,51,0,16,0,0,...,No,No,No,No,Ch,Yes,>30,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY
101762,74694222,Female,[80-90),5,Government,33,3,18,0,0,...,No,No,No,No,No,Yes,NO,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER
101763,41088789,Male,[70-80),1,Government,53,0,9,1,0,...,No,No,No,No,Ch,Yes,NO,REFERRAL,HOME,EMERGENCY
101764,31693671,Female,[80-90),10,Government,45,2,21,0,0,...,No,No,No,No,Ch,Yes,NO,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY


In [20]:
# remove those admitted after 30 days
df_clean = df_clean[~(df_clean.readmitted == '>30')]

In [21]:
df_clean.readmitted.value_counts()

readmitted
NO     54256
<30    11275
Name: count, dtype: int64

In [22]:
df_clean['readmitted'] = df_clean['readmitted'].map({'NO': 0, '<30': 1})

C:\Users\sar31\AppData\Local\Temp\ipykernel_13100\2508839561.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['readmitted'] = df_clean['readmitted'].map({'NO': 0, '<30': 1})


In [23]:
df_clean

,patient_nbr,gender,age,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped
2,86047875,Female,[20-30),2,Unknown,11,5,13,2,0,...,No,No,No,No,No,Yes,0,REFERRAL,HOME,EMERGENCY
3,82442376,Male,[30-40),2,Unknown,44,1,16,0,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,HOME,EMERGENCY
4,42519267,Male,[40-50),1,Unknown,51,0,8,0,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,HOME,EMERGENCY
6,84259809,Male,[60-70),4,Unknown,70,1,21,0,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,HOME,REFERRAL
8,48330783,Female,[80-90),13,Unknown,68,2,28,0,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,HOME,TRANSFER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,86472243,Male,[80-90),1,Government,1,0,15,3,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,HOME,EMERGENCY
101762,74694222,Female,[80-90),5,Government,33,3,18,0,0,...,No,No,No,No,No,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER
101763,41088789,Male,[70-80),1,Government,53,0,9,1,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,HOME,EMERGENCY
101764,31693671,Female,[80-90),10,Government,45,2,21,0,0,...,No,No,No,No,Ch,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY


In [24]:
medicines_df = df_clean[['max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone']].copy()
medicines_df

,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,...,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone
2,NaN,NaN,No,No,No,No,No,No,Steady,No,...,No,No,No,No,No,No,No,No,No,No
3,NaN,NaN,No,No,No,No,No,No,No,No,...,No,No,No,No,Up,No,No,No,No,No
4,NaN,NaN,No,No,No,No,No,No,Steady,No,...,No,No,No,No,Steady,No,No,No,No,No
6,NaN,NaN,Steady,No,No,No,Steady,No,No,No,...,No,No,No,No,Steady,No,No,No,No,No
8,NaN,NaN,No,No,No,No,No,No,Steady,No,...,No,No,No,No,Steady,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,NaN,NaN,No,No,No,No,No,No,No,No,...,No,No,No,No,Up,No,No,No,No,No
101762,NaN,NaN,No,No,No,No,No,No,No,No,...,No,No,No,No,Steady,No,No,No,No,No
101763,NaN,NaN,Steady,No,No,No,No,No,No,No,...,No,No,No,No,Down,No,No,No,No,No
101764,NaN,NaN,No,No,No,No,No,No,Steady,No,...,No,No,No,No,Up,No,No,No,No,No


In [25]:
# each medication has No, Steady, Up and Down as values

In [26]:
medicines_df.isna().mean() # drop max_glu_serum and A1Cresult

max_glu_serum               0.948345
A1Cresult                   0.833789
metformin                   0.000000
repaglinide                 0.000000
nateglinide                 0.000000
chlorpropamide              0.000000
glimepiride                 0.000000
acetohexamide               0.000000
glipizide                   0.000000
glyburide                   0.000000
tolbutamide                 0.000000
pioglitazone                0.000000
rosiglitazone               0.000000
acarbose                    0.000000
miglitol                    0.000000
troglitazone                0.000000
tolazamide                  0.000000
examide                     0.000000
citoglipton                 0.000000
insulin                     0.000000
glyburide-metformin         0.000000
glipizide-metformin         0.000000
glimepiride-pioglitazone    0.000000
metformin-rosiglitazone     0.000000
metformin-pioglitazone      0.000000
dtype: float64

In [27]:
df_clean = df_clean.drop(columns=['A1Cresult', 'max_glu_serum'])

In [28]:
# use of medication by encounter, how many times was this medication prescribed ? that is the dose is not NO. 
(medicines_df != 'No').mean().sort_values(ascending=False)

max_glu_serum               1.000000
A1Cresult                   1.000000
insulin                     0.522486
metformin                   0.201813
glipizide                   0.121271
glyburide                   0.105507
pioglitazone                0.069677
rosiglitazone               0.060201
glimepiride                 0.050068
repaglinide                 0.013703
glyburide-metformin         0.006836
nateglinide                 0.006714
acarbose                    0.002335
chlorpropamide              0.000794
tolazamide                  0.000443
miglitol                    0.000275
tolbutamide                 0.000244
glipizide-metformin         0.000092
metformin-rosiglitazone     0.000031
metformin-pioglitazone      0.000015
troglitazone                0.000015
citoglipton                 0.000000
examide                     0.000000
glimepiride-pioglitazone    0.000000
acetohexamide               0.000000
dtype: float64

In [29]:
# List of all medication columns
medication_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 
    'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
    'metformin-pioglitazone'
]

meds_to_keep = ["insulin", "metformin", "glipizide", "glyburide", "pioglitazone", "rosiglitazone", "glimepiride"]
medications_to_drop = [ 'repaglinide', 'nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'tolazamide', 'examide', 'citoglipton', 'troglitazone', 'acarbose', 'miglitol', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

# Create med_count: count how many medications were prescribed (not "No")
df_clean['med_count'] = (df[medication_columns] != 'No').sum(axis=1)

df_clean = df_clean.drop(columns=medications_to_drop)

In [30]:
df_clean

,patient_nbr,gender,age,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,pioglitazone,rosiglitazone,insulin,change,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped,med_count
2,86047875,Female,[20-30),2,Unknown,11,5,13,2,0,...,No,No,No,No,Yes,0,REFERRAL,HOME,EMERGENCY,1
3,82442376,Male,[30-40),2,Unknown,44,1,16,0,0,...,No,No,Up,Ch,Yes,0,REFERRAL,HOME,EMERGENCY,1
4,42519267,Male,[40-50),1,Unknown,51,0,8,0,0,...,No,No,Steady,Ch,Yes,0,REFERRAL,HOME,EMERGENCY,2
6,84259809,Male,[60-70),4,Unknown,70,1,21,0,0,...,No,No,Steady,Ch,Yes,0,REFERRAL,HOME,REFERRAL,3
8,48330783,Female,[80-90),13,Unknown,68,2,28,0,0,...,No,No,Steady,Ch,Yes,0,REFERRAL,HOME,TRANSFER,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,86472243,Male,[80-90),1,Government,1,0,15,3,0,...,No,No,Up,Ch,Yes,0,REFERRAL,HOME,EMERGENCY,1
101762,74694222,Female,[80-90),5,Government,33,3,18,0,0,...,No,No,Steady,No,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER,1
101763,41088789,Male,[70-80),1,Government,53,0,9,1,0,...,No,No,Down,Ch,Yes,0,REFERRAL,HOME,EMERGENCY,2
101764,31693671,Female,[80-90),10,Government,45,2,21,0,0,...,Steady,No,Up,Ch,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY,3


In [31]:
# I am also dropping the change variable because I can derive based on Up/Down if the medication was changed. Yes there will be the rare case when there has been any change in the medicines that I have dropped.
df_clean = df_clean.drop(columns='change')

In [32]:
# process diag_1, diag_2, and diag_3

df_clean['diag_1_cat'] = df_clean['diag_1'].apply(categorize_diagnosis)
df_clean['diag_2_cat'] = df_clean['diag_2'].apply(categorize_diagnosis)
df_clean['diag_3_cat'] = df_clean['diag_3'].apply(categorize_diagnosis)

# Drop original diagnosis columns
df_clean = df_clean.drop(columns=['diag_1', 'diag_2', 'diag_3'])

In [33]:
# make age into categorical ordered variable
df_clean

,patient_nbr,gender,age,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,insulin,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped,med_count,diag_1_cat,diag_2_cat,diag_3_cat
2,86047875,Female,[20-30),2,Unknown,11,5,13,2,0,...,No,Yes,0,REFERRAL,HOME,EMERGENCY,1,Other,Diabetes,Other
3,82442376,Male,[30-40),2,Unknown,44,1,16,0,0,...,Up,Yes,0,REFERRAL,HOME,EMERGENCY,1,Other,Diabetes,Circulatory
4,42519267,Male,[40-50),1,Unknown,51,0,8,0,0,...,Steady,Yes,0,REFERRAL,HOME,EMERGENCY,2,Neoplasms,Neoplasms,Diabetes
6,84259809,Male,[60-70),4,Unknown,70,1,21,0,0,...,Steady,Yes,0,REFERRAL,HOME,REFERRAL,3,Circulatory,Circulatory,Other
8,48330783,Female,[80-90),13,Unknown,68,2,28,0,0,...,Steady,Yes,0,REFERRAL,HOME,TRANSFER,2,Circulatory,Circulatory,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,86472243,Male,[80-90),1,Government,1,0,15,3,0,...,Up,Yes,0,REFERRAL,HOME,EMERGENCY,1,Circulatory,Other,Diabetes
101762,74694222,Female,[80-90),5,Government,33,3,18,0,0,...,Steady,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER,1,Digestive,Other,Digestive
101763,41088789,Male,[70-80),1,Government,53,0,9,1,0,...,Down,Yes,0,REFERRAL,HOME,EMERGENCY,2,Other,Genitourinary,Other
101764,31693671,Female,[80-90),10,Government,45,2,21,0,0,...,Up,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY,3,Injury,Other,Injury


In [34]:
# Define the mapping of age labels to numeric values
age_mapping = { '[20-30)': 1, '[30-40)': 2, '[40-50)': 3, '[50-60)': 4,
    '[60-70)': 5, '[70-80)': 6, '[80-90)': 7, '[90-100)': 8
}

# Apply the mapping
df_clean['age'] = df_clean['age'].map(age_mapping)

# make categorical
df_clean['age'] = pd.Categorical(df_clean.age, categories=[1, 2, 3, 4, 5, 6, 7, 8], ordered=True)

In [35]:
df_clean

,patient_nbr,gender,age,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,insulin,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped,med_count,diag_1_cat,diag_2_cat,diag_3_cat
2,86047875,Female,1,2,Unknown,11,5,13,2,0,...,No,Yes,0,REFERRAL,HOME,EMERGENCY,1,Other,Diabetes,Other
3,82442376,Male,2,2,Unknown,44,1,16,0,0,...,Up,Yes,0,REFERRAL,HOME,EMERGENCY,1,Other,Diabetes,Circulatory
4,42519267,Male,3,1,Unknown,51,0,8,0,0,...,Steady,Yes,0,REFERRAL,HOME,EMERGENCY,2,Neoplasms,Neoplasms,Diabetes
6,84259809,Male,5,4,Unknown,70,1,21,0,0,...,Steady,Yes,0,REFERRAL,HOME,REFERRAL,3,Circulatory,Circulatory,Other
8,48330783,Female,7,13,Unknown,68,2,28,0,0,...,Steady,Yes,0,REFERRAL,HOME,TRANSFER,2,Circulatory,Circulatory,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,86472243,Male,7,1,Government,1,0,15,3,0,...,Up,Yes,0,REFERRAL,HOME,EMERGENCY,1,Circulatory,Other,Diabetes
101762,74694222,Female,7,5,Government,33,3,18,0,0,...,Steady,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER,1,Digestive,Other,Digestive
101763,41088789,Male,6,1,Government,53,0,9,1,0,...,Down,Yes,0,REFERRAL,HOME,EMERGENCY,2,Other,Genitourinary,Other
101764,31693671,Female,7,10,Government,45,2,21,0,0,...,Up,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY,3,Injury,Other,Injury


In [36]:
# Columns to exclude these and those that start with number_ or num_
exclude_columns = ['patient_nbr', 'age', 'time_in_hospital', 'readmitted']

# identify the columns to create dummies for 
dummy_cols = [col for col in df_clean.columns if col not in exclude_columns and not col.startswith(('num_', 'number_'))]
dummy_cols

['gender',
 'payer_code',
 'metformin',
 'glimepiride',
 'glipizide',
 'glyburide',
 'pioglitazone',
 'rosiglitazone',
 'insulin',
 'diabetesMed',
 'admission_type_grouped',
 'discharge_disposition_grouped',
 'admission_source_grouped',
 'med_count',
 'diag_1_cat',
 'diag_2_cat',
 'diag_3_cat']

In [37]:
df_clean.gender.value_counts()

gender
Female             34818
Male               30710
Unknown/Invalid        3
Name: count, dtype: int64

In [38]:
# drop the 3 unknown rows
df_clean = df_clean[~(df_clean.gender == 'Unknown/Invalid')]

In [39]:
df_clean.payer_code.value_counts()

payer_code
Unknown         27745
Government      23546
Private         11017
Self-Pay         3109
Workers-Comp      111
Name: count, dtype: int64

In [40]:
df_clean

,patient_nbr,gender,age,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,insulin,diabetesMed,readmitted,admission_type_grouped,discharge_disposition_grouped,admission_source_grouped,med_count,diag_1_cat,diag_2_cat,diag_3_cat
2,86047875,Female,1,2,Unknown,11,5,13,2,0,...,No,Yes,0,REFERRAL,HOME,EMERGENCY,1,Other,Diabetes,Other
3,82442376,Male,2,2,Unknown,44,1,16,0,0,...,Up,Yes,0,REFERRAL,HOME,EMERGENCY,1,Other,Diabetes,Circulatory
4,42519267,Male,3,1,Unknown,51,0,8,0,0,...,Steady,Yes,0,REFERRAL,HOME,EMERGENCY,2,Neoplasms,Neoplasms,Diabetes
6,84259809,Male,5,4,Unknown,70,1,21,0,0,...,Steady,Yes,0,REFERRAL,HOME,REFERRAL,3,Circulatory,Circulatory,Other
8,48330783,Female,7,13,Unknown,68,2,28,0,0,...,Steady,Yes,0,REFERRAL,HOME,TRANSFER,2,Circulatory,Circulatory,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,86472243,Male,7,1,Government,1,0,15,3,0,...,Up,Yes,0,REFERRAL,HOME,EMERGENCY,1,Circulatory,Other,Diabetes
101762,74694222,Female,7,5,Government,33,3,18,0,0,...,Steady,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,TRANSFER,1,Digestive,Other,Digestive
101763,41088789,Male,6,1,Government,53,0,9,1,0,...,Down,Yes,0,REFERRAL,HOME,EMERGENCY,2,Other,Genitourinary,Other
101764,31693671,Female,7,10,Government,45,2,21,0,0,...,Up,Yes,0,REFERRAL,TRANSFERRED_POST_ACUTE,EMERGENCY,3,Injury,Other,Injury


In [41]:
# convert the columns to one hot
df_dummies = []
for col in dummy_cols:
    df_dummies.append(pd.get_dummies(df_clean[col], drop_first=True, prefix=col))

df_dummies_concat = pd.concat(df_dummies, axis='columns')
df_dummies_concat

,gender_Male,payer_code_Private,payer_code_Self-Pay,payer_code_Unknown,payer_code_Workers-Comp,metformin_No,metformin_Steady,metformin_Up,glimepiride_No,glimepiride_Steady,...,diag_2_cat_Respiratory,diag_3_cat_Diabetes,diag_3_cat_Digestive,diag_3_cat_Genitourinary,diag_3_cat_Injury,diag_3_cat_Missing,diag_3_cat_Musculoskeletal,diag_3_cat_Neoplasms,diag_3_cat_Other,diag_3_cat_Respiratory
2,False,False,False,True,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,True,False
3,True,False,False,True,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,True,False,True,False,False,True,False,...,False,True,False,False,False,False,False,False,False,False
6,True,False,False,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
8,False,False,False,True,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101759,True,False,False,False,False,True,False,False,True,False,...,False,True,False,False,False,False,False,False,False,False
101762,False,False,False,False,False,True,False,False,True,False,...,False,False,True,False,False,False,False,False,False,False
101763,True,False,False,False,False,False,True,False,True,False,...,False,False,False,False,False,False,False,False,True,False
101764,False,False,False,False,False,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False


In [42]:
df_clean = pd.concat([df_clean.drop(columns=dummy_cols), df_dummies_concat], axis='columns')

In [43]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 65528 entries, 2 to 101765
Data columns (total 82 columns):
 #   Column                                                Non-Null Count  Dtype   
---  ------                                                --------------  -----   
 0   patient_nbr                                           65528 non-null  int64   
 1   age                                                   65528 non-null  category
 2   time_in_hospital                                      65528 non-null  int64   
 3   num_lab_procedures                                    65528 non-null  int64   
 4   num_procedures                                        65528 non-null  int64   
 5   num_medications                                       65528 non-null  int64   
 6   number_outpatient                                     65528 non-null  int64   
 7   number_emergency                                      65528 non-null  int64   
 8   number_inpatient                                  

In [ ]:
df_clean.isna().sum().sum() # luckily there are no NA values to impute or take care of. 

0

In [44]:
df_clean.to_parquet(data_dir / "cleaned_data.parquet")